Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en tres partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

    
- [Actividad 1: Redes Recurrentes](#actividad_1): 10 pts
    - [Cuestión 1](#3.1): 2.5 pt
    - [Cuestión 2](#3.2): 2.5 pt
    - [Cuestión 3](#3.3): 2.5 pts
    - [Cuestión 4](#3.4): 1.25 pts
    - [Cuestión 5](#3.5): 1.25 pts



In [34]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

<a name='actividad_1'></a>
# Actividad 1: Redes Recurrentes


- [Cuestión 1](#3.1): 2.5 pt
- [Cuestión 2](#3.2): 2.5 pt
- [Cuestión 3](#3.3): 2.5 pts
- [Cuestión 4](#3.4): 1.25 pts
- [Cuestión 5](#3.5): 1.25 pts

Vamos a usar un dataset de las temperaturas mínimas diarias en Melbourne. La tarea será la de predecir la temperatura mínima en dos días. Puedes usar técnicas de series temporales vistas en otras asignaturas, pero no es necesario.


In [35]:
dataset_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
data_dir = tf.keras.utils.get_file('daily-min-temperatures.csv', origin=dataset_url)

In [36]:
df = pd.read_csv(data_dir, parse_dates=['Date'])
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [37]:
temperatures = df['Temp'].values
print('number of samples:', len(temperatures))
train_data = temperatures[:3000]
test_data = temperatures[3000:]
print('number of train samples:', len(train_data))
print('number of test samples:', len(test_data))
print('firsts trainn samples:', train_data[:10])

number of samples: 3650
number of train samples: 3000
number of test samples: 650
firsts trainn samples: [20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]


<a name='3.1'></a>
## Cuestión 1: Convierta `train_data` y `test_data`  en ventanas de tamaño 5, para predecir el valor en 2 días

En la nomenclatura de [Introduction_to_RNN_Time_Series.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/RNN/Introduction_to_RNN_Time_Series.ipynb)
```python
past, future = (5, 2)
```

Para las primeras 10 muestras de train_data `[20.7, 17.9, 18.8, 14.6, 15.8, 15.8, 15.8, 17.4, 21.8, 20. ]` el resultado debería ser:

```python
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.             
```

In [38]:
# windowing function
def create_windows_np(data, window_size, horizon, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.

    Parameters:
    data (np.ndarray): Time series data with one dimension.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    shuffle (bool): Shuffle the windows or not.

    Returns:
    tuple: A tuple containing the input-output pairs (windows, targets) as NumPy arrays.
    """

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size+horizon-1])

    X, y = np.array(X), np.array(y)

    if shuffle:
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]

    return X, y

In [39]:
past, future = (5, 2)
X_train, y_train = create_windows_np(train_data, past, future, shuffle=False)
X_test, y_test = create_windows_np(test_data, past, future, shuffle=False)

In [40]:
X_train.shape

(2994, 5)

<a name='3.2'></a>
## Cuestión 2: Cree un modelo recurrente de dos capas GRU para predecir con las ventanas de la cuestión anterior.


In [ ]:
inputs = keras.layers.Input(shape=(past, 1))

gru1 = layers.GRU(64, return_sequences=True, name='gru_1')(inputs)
gru2 = layers.GRU(64, return_sequences=False, name='gru_2')(gru1)
outputs = keras.layers.Dense(1)(gru2)

model = keras.Model(inputs=inputs, outputs=outputs, name='gru_model')
model.compile(optimizer=keras.optimizers.Adam(), loss="mse")
model.summary()

Model: "gru_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 5, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 5, 64)          │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,889 (148.00 KB)

 Trainable params: 37,889 (148.00 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - loss: 96.7056 - val_loss: 24.5790
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 17.8992 - val_loss: 17.6215
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 15.8634 - val_loss: 16.3904
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 15.0543 - val_loss: 13.3482
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 12.0745 - val_loss: 10.8587
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 10.1994 - val_loss: 10.5148
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 9.9299 - val_loss: 9.1248
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 9.6255 - val_loss: 8.7297
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.7085 - val_loss: 9.1753
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.4075 - val_loss: 8.9315
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 8.6853 - val_loss: 8.6244
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 

In [43]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 6.7926
Test Loss: 7.052270889282227


<a name='3.3'></a>
## Cuestión 3: Añada más features a la series temporal, por ejemplo `portion_year`. Cree un modelo que mejore al anterior.


In [55]:
## Puede añadir más features
df['portion_year'] = df['Date'].dt.dayofyear / 365.0
df_multi = df[['Temp', 'portion_year']].copy()

## train - test split
train_data = df_multi.iloc[:3000].copy()
test_data = df_multi.loc[3000:, :].copy()

In [56]:
def create_windows_multivariate_np(data, window_size, horizon, target_col_idx, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.

    Parameters:
    data (np.ndarray or pd.DataFrame): Time series data with multiple features.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    target_col_idx (int): The index of the target column in the input data.
    shuffle (bool): Whether to shuffle the data or not.

    Returns:
    triplet: A triplet containing the input-output pairs (X, y) as NumPy arrays.
    """
    if isinstance(data, pd.DataFrame):
        data = data.values

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i + window_size, :])
        y.append(data[i + window_size:i + window_size + horizon, target_col_idx])

    X, y = np.array(X), np.array(y)

    if shuffle:
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]

    return X, y

In [57]:
## Create windows
past, future = (5, 2)
target_col_idx = 0 # Índice de la columna Temp (la que queremos predecir)

X_train, y_train = create_windows_multivariate_np(train_data, past, future,target_col_idx, shuffle=False)
X_test, y_test = create_windows_multivariate_np(test_data, past, future,target_col_idx, shuffle=False)

In [86]:
inputs = keras.layers.Input(shape=(past, 2))
gru1 = keras.layers.GRU(64, return_sequences=True)(inputs)
gru2 = keras.layers.GRU(64, return_sequences=False)(gru1)
outputs = keras.layers.Dense(future)(gru2)

model = keras.Model(inputs, outputs, name='gru_model_2')
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "gru_model_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_20 (InputLayer)     │ (None, 5, 2)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_41 (GRU)                    │ (None, 5, 64)          │        13,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_42 (GRU)                    │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,146 (149.01 KB)

 Trainable params: 38,146 (149.01 KB)

 Non-trainable params: 0 (0.00 B)

In [87]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 92.3737 - mae: 8.5115 - val_loss: 41.7027 - val_mae: 5.3838
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 28.1444 - mae: 4.2473 - val_loss: 25.3107 - val_mae: 4.0287
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 19.7384 - mae: 3.5376 - val_loss: 19.5079 - val_mae: 3.5358
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 17.5212 - mae: 3.3378 - val_loss: 17.0858 - val_mae: 3.2926
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 13.8969 - mae: 2.9176 - val_loss: 13.5596 - val_mae: 2.8626
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 10.6839 - mae: 2.5242 - val_loss: 11.1212 - val_mae: 2.5786
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 9.4705 - mae: 2.3673 - val_loss: 9.7689 - val_mae: 2.4067
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 8.7764 - mae: 2.2580 - val_loss: 8.9427 - val_mae: 2.3081
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 25

In [88]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5.4476 - mae: 1.8241
Test Loss: [5.761125564575195, 1.8665722608566284]


<a name='3.4'></a>
## Cuestión 4: ¿En cuáles de estas aplicaciones se usaría un arquitectura 'many-to-one'?

**a)** Clasificación de sentimiento en textos

**b)** Verificación de voz para iniciar el ordenador.

**c)** Generación de música.

**d)** Un clasificador que clasifique piezas de música según su autor.


a), b) y d)

<a name='3.5'></a>
## Cuestión 5: ¿Qué ventajas aporta el uso de word embeddings?

**a)** Permiten reducir la dimensión de entrada respecto al one-hot encoding.

**b)** Permiten descubrir la similaridad entre palabras de manera más intuitiva que con one-hot encoding.

**c)** Son una manera de realizar transfer learning en nlp.

**d)** Permiten visualizar las relaciones entre palabras con métodos de reducción de dimensioones como el PCA.


a), b), c) y d)